In [ ]:
import requests
import re
import multithreading
from bs4 import BeautifulSoup as bs
from utils.exceptions import *

In [2]:
url = 'https://esaj.tjsp.jus.br/cjpg/pesquisar.do?conversationId=&dadosConsulta.pesquisaLivre=&tipoNumero=UNIFICADO&numeroDigitoAnoUnificado=&foroNumeroUnificado=&dadosConsulta.nuProcesso=&dadosConsulta.nuProcessoAntigo=&classeTreeSelection.values=&classeTreeSelection.text=&assuntoTreeSelection.values=15042&assuntoTreeSelection.text=Despejo+por+Inadimplemento&agenteSelectedEntitiesList=&contadoragente=0&contadorMaioragente=0&cdAgente=&nmAgente=&dadosConsulta.dtInicio=&dadosConsulta.dtFim=&varasTreeSelection.values=&varasTreeSelection.text=&dadosConsulta.ordenacao=DESC'

In [ ]:
def start_search_session():
    session = requests.Session()
    session.get(url)
    return session

def session_get_page(page, session=None):
    if not isinstance(session, requests.Session):
        session = start_search_session()
    page_url = f'https://esaj.tjsp.jus.br/cjpg/trocarDePagina.do?pagina={page}&conversationId='
    response = session.get(page_url)
    return response

def get_page_counts(soup):
    count_td = clean_string(soup.find('td').text)
    page_min, page_max, page_sentence_count = [int(i) for i in re.findall(r'\d+', count_td)]
    return page_min, page_max, page_sentence_count

def check_response(response, min_sentence_count=0):
    soup = bs(response.text)

    # Check for expired session
    if 'Sessão Expirada' in clean_string(soup.text):
        raise ExpiredSession
    
    # Check sentences count
    try:
        _, _, page_sentence_count = get_page_counts(soup)
    except:
        raise ExpiredSession
    if page_sentence_count < min_sentence_count:
        raise CountBelowMinimum
    
def clean_string(string):
    # Limpeza de cariage return e form feed
    while '\r' in string:
        string = string.replace('\r', '')
    while '\f' in string:
        string = string.replace('\f', '')

    # Troca de tabulações por espaços e remoção de espaços duplos
    string = string.replace('\t', ' ')
    while '  ' in string:
        string = string.replace('  ', ' ')

    # Limpeza de linhas
    while '\n ' in string:
        string = string.replace('\n ', '\n')
    while '\n\n' in string:
        string = string.replace('\n\n', '\n')
    
    return string.strip()

def get_response_from_page(page, max_attempts=100):
    # Tries to get a valid soup
    session = start_search_session()
    attempts = 0
    while attempts < max_attempts:
        try:
            response = session_get_page(page, session)
            check_response(response)
            return response
        except ExpiredSession:
            session.close()
            session = start_search_session()
            attempts+=1
            continue
        except CountBelowMinimum:
            attempts+=1
            continue
    raise MaxAttemptsReached

def get_min_sentence_count(max_attempts=50):
    min_sentence_count = 0
    count = 0
    while count < max_attempts:
        try:
            response = get_response_from_page(1, max_attempts=max_attempts)
            soup = bs(response.text)
            _, _, page_sentence_count = get_page_counts(soup)
            if page_sentence_count > min_sentence_count == 0:
                min_sentence_count = page_sentence_count
            count+=1
        except MaxAttemptsReached:
            print('error')
            raise
    return min_sentence_count

In [31]:
# Ler primeira página e obter máximos de pesquisa
min_sentence_count = get_min_sentence_count()

# Criar fila com páginas a serem raspadas

# Raspar páginas e salvar incrementalmente

In [32]:
min_sentence_count

83717

In [ ]:
def worker(queue)
get_soup_from_page(3)


<table border="0" cellpadding="0" cellspacing="0" width="100%">
<tr height="20">
<td bgcolor="#EEEEEE">
			Resultados

			<strong>
				21 a 30
			</strong>
			de 83687
		</td>
<td align="right" bgcolor="#EEEEEE">
<div class="trocaDePagina">
<a name="2" title="Página anterior">
				&lt;
			</a>
<a href="#" name="1" style="padding-left:4px">
						1
					</a>
<a href="#" name="2" style="padding-left:4px">
						2
					</a>
<span style="font-weight:bold;">
						3
					</span>
<a href="#" name="4" style="padding-left:4px">
						4
					</a>
<a href="#" name="5" style="padding-left:4px">
						5
					</a>
<a name="4" title="Próxima página">
				&gt;
			</a>
</div>
</td>
</tr>
<tr>
<td bgcolor="#999999" colspan="2" height="1"></td>
</tr>
</table>
 

<table border="0" cellpadding="0" cellspacing="0" width="100%">
<tr>
<td height="1" id="tdResultados" valign="top" width="*">
<div id="divDadosResultado">
<table cellpadding="0" cellspacing="0" class="" width="100%">
<tr class="fundocinza1">
<t

In [22]:
def ex():
    raise Exception
    return 1